### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 업스테이지의 콘텐츠 자산으로, 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.

# 8강 실습 — Production Reliability Layer

> 목표: 자율 에이전트가 **외부 실패에도 안 무너지도록** 4대 신뢰성 패턴을 직접 구현해 씌워봅니다 —
> **Circuit Breaker · Retry(+Jitter) · Budget · Idempotency**.
>
> 산출물: 각 패턴 구현 + 통합 Reliability Layer로 감싼 도구 호출

## 0. 셋업

In [40]:
# ── 셋업 ──
import time, random, hashlib, json
random.seed(7)
print("준비 완료")

준비 완료


## 1. Circuit Breaker — 두꺼비집
실패율이 임계를 넘으면 **OPEN**으로 차단해 장애 확산을 막고, 일정 시간 뒤 **HALF-OPEN**으로 시험 복구합니다.

In [41]:
# ── Circuit Breaker — 두꺼비집 ──

class CircuitBreaker:
    def __init__(self, fail_threshold=3, reset_timeout=2.0):
        self.fail_threshold=fail_threshold; self.reset_timeout=reset_timeout
        self.fails=0; self.state="CLOSED"; self.opened_at=0
    def call(self, fn, *a, **k):
        if self.state=="OPEN":
            if time.time()-self.opened_at > self.reset_timeout:
                self.state="HALF-OPEN"; print("   ↪ HALF-OPEN (시험 복구)")
            else:
                raise RuntimeError("CircuitOpen — 호출 차단")
        try:
            r=fn(*a,**k)
            if self.state=="HALF-OPEN": self.state="CLOSED"; self.fails=0; print("   ✅ 복구 → CLOSED")
            return r
        except Exception:
            self.fails+=1
            if self.fails>=self.fail_threshold:
                self.state="OPEN"; self.opened_at=time.time(); print(f"   🔌 임계 도달 → OPEN")
            raise

calls={"n":0}
def flaky_api():
    calls["n"]+=1
    if calls["n"]<=4: raise TimeoutError("504")
    return "OK"

cb=CircuitBreaker(fail_threshold=3, reset_timeout=1.0)
for i in range(5):
    try: print(f"[{i+1}] ", cb.call(flaky_api), "| state=",cb.state)
    except Exception as e: print(f"[{i+1}]  실패: {e} | state=",cb.state)
time.sleep(1.1)
print("타임아웃 경과 후 재시도:")
for i in range(2):
    try: print("  ", cb.call(flaky_api), "| state=",cb.state)
    except Exception as e: print("  실패:",e)

[1]  실패: 504 | state= CLOSED
[2]  실패: 504 | state= CLOSED
   🔌 임계 도달 → OPEN
[3]  실패: 504 | state= OPEN
[4]  실패: CircuitOpen — 호출 차단 | state= OPEN
[5]  실패: CircuitOpen — 호출 차단 | state= OPEN
타임아웃 경과 후 재시도:
   ↪ HALF-OPEN (시험 복구)
   🔌 임계 도달 → OPEN
  실패: 504
  실패: CircuitOpen — 호출 차단


## 2. Retry + Exponential Backoff + Jitter
일시적 실패는 **지수 백오프 + 지터**로 재시도 하며, 지터가 동시 폭주를 막습니다.

In [42]:
# ── Retry + Exponential Backoff + Jitter ──
def retry(fn, max_retries=3, base=0.2, jitter=True):
    for attempt in range(max_retries+1):
        try: return fn()
        except Exception as e:
            if attempt==max_retries:
              raise
            delay=base*(2**attempt)*((0.5+random.random()) if jitter else 1)
            # delay = 0.2 * (2**0) & (0.6) ->  0.2*0.6 = 0.12 # 1번째 retry
            # delay = 0.2 * (2**1) & (0.6) ->  0.2*0.6*2 = 0.24 # 1번째 retry
            # ..
            # delay = 0.2 * (2**10) & (0.6) ->  0.2*0.6*1024 = ~ 120초 # 1번째 retry
            print(f"   재시도 {attempt+1} (대기 {delay:.2f}s) — {e}")
            time.sleep(min(delay,0.5))
calls["n"]=0
print("결과:", retry(flaky_api, max_retries=5))

   재시도 1 (대기 0.16s) — 504
   재시도 2 (대기 0.26s) — 504
   재시도 3 (대기 0.92s) — 504
   재시도 4 (대기 0.92s) — 504
결과: OK


## 3. Budget — 스텝/토큰 예산
무한 루프·비용 폭주를 방지하며, 예산 초과 시 즉시 중단합니다.

In [45]:
# ── Budget — 스텝/토큰 예산 ──

class Budget:
    def __init__(self, max_steps=5, max_tokens=10000): self.steps=max_steps; self.tokens=max_tokens
    def step(self, used_tokens=0):
        self.steps= self.steps-1; self.tokens-=used_tokens
        print(self.steps, self.tokens)
        if self.steps<0: raise RuntimeError("StepBudget 초과")
        if self.tokens<0: raise RuntimeError("TokenBudget 초과")
b=Budget(max_steps=3)
try:
    for i in range(5): b.step(); print(f"   step {i+1} OK (남은 {b.steps})")
except RuntimeError as e: print("   🛑", e)

2 10000
   step 1 OK (남은 2)
1 10000
   step 2 OK (남은 1)
0 10000
   step 3 OK (남은 0)
-1 10000
   🛑 StepBudget 초과


## 4. Idempotency — 중복 실행 방지
재시도가 **부수효과 작업(송금·이메일)**을 중복 실행하지 않도록 멱등 키(Idempotency Key)로 1회 보장합니다.

In [48]:
# ── Idempotency — 중복 실행 방지 ──

_done={}
def idem_key(op, params): return hashlib.sha256(f"{op}:{json.dumps(params,sort_keys=True)}".encode()).hexdigest()[:12]
def send_email_idem(to, subject):
    k=idem_key("send_email",{"to":to,"subject":subject})
    if k in _done: return f"이미 발송됨(캐시) [{k}]"
    _done[k]={"to":to}; return f"발송 [{k}]"
print(send_email_idem("a@x.com","환불완료"))
print(send_email_idem("a@x.com","환불완료"))  # 재시도 → 중복 안 됨
print(send_email_idem("a@x.com","환불완료-수정"))

발송 [c6d3fa91116c]
이미 발송됨(캐시) [c6d3fa91116c]
발송 [e2f0f772709f]


## 5. 샌드박스 실행 — 위험 코드 차단
에이전트가 생성한 코드를 **제한된 빌트인**으로만 실행해 파일 삭제·임의 import 같은 위험을 막고, 무한 루프는 시간 예산으로 끊습니다.

In [49]:
import time, signal

# 1. 시간 제한용 예외 클래스 정의
class TimeBudgetExceeded(Exception): pass

# 2. Context Manager를 이용한 시간 예산 가드
class time_budget:
    def __init__(self, limit):
        self.limit = limit
    def __enter__(self):
        def _raise(signum, frame): raise TimeBudgetExceeded(f"⏱ {self.limit}s 초과")
        self._old = signal.signal(signal.SIGALRM, _raise)
        signal.setitimer(signal.ITIMER_REAL, self.limit)
        return self
    def __exit__(self, *a):
        signal.setitimer(signal.ITIMER_REAL, 0)
        signal.signal(signal.SIGALRM, self._old)
        return False

# 3. 안전한 빌트인 함수만 허용 (화이트리스트)
_B = __builtins__ if isinstance(__builtins__, dict) else vars(__builtins__)
SAFE_BUILTINS = {k: _B[k] for k in
    ["range","len","sum","min","max","print","abs","sorted","list","dict","int","float","str"]}

def run_in_sandbox(code, limit=2.0):
    """제한된 환경에서 코드를 실행하는 샌드박스 함수"""
    g = {"__builtins__": SAFE_BUILTINS}
    try:
        with time_budget(limit):
            exec(code, g)
        return g.get("result", "(결과값 result 변수 없음)")
    except TimeBudgetExceeded:
        return "🛑 시간 초과(무한 루프 의심) — 실행 중단"
    except Exception as e:
        return f"🛑 보안 차단됨: {type(e).__name__}: {e}"

# 테스트 코드 실행
print("--- 샌드박스 테스트 ---")
print("1. 안전 코드:", run_in_sandbox("result = sum(range(10))"))
print("2. 위험 코드(os 임포트):", run_in_sandbox("import os; os.remove('/etc/hosts')"))
print("3. 무한 루프:", run_in_sandbox("while True: pass", limit=0.5))

--- 샌드박스 테스트 ---
1. 안전 코드: 45
2. 위험 코드(os 임포트): 🛑 보안 차단됨: ImportError: __import__ not found
3. 무한 루프: 🛑 시간 초과(무한 루프 의심) — 실행 중단


## 6. 통합 Reliability Layer — 모든 도구 호출을 감싸기
회로 차단기 · 재시도 · 멱등성 · 예산, Circuit Breaker + Retry 를 **하나의 호출 경로**로 묶습니다. 도구 하나가 불안정해도 전체가 안전하게 동작합니다.


In [50]:
import time, random

class Budget:
    """토큰과 실행 단계(Step) 예산을 관리하는 클래스"""
    def __init__(self, max_tokens, max_steps):
        self.max_tokens, self.max_steps = max_tokens, max_steps
        self.tokens = self.steps = 0
    def charge(self, tokens):
        self.steps += 1; self.tokens += tokens
        if self.steps > self.max_steps: raise RuntimeError(f"Budget 초과: 스텝 {self.steps}")
        if self.tokens > self.max_tokens: raise RuntimeError(f"Budget 초과: 토큰 {self.tokens}")

class CircuitBreaker:
    """연속 실패 시 호출을 즉시 차단하는 회로 차단기"""
    def __init__(self, fail_threshold=3, reset_timeout=0.5):
        self.fail_threshold, self.reset_timeout = fail_threshold, reset_timeout
        self.fails = 0; self.state = "CLOSED"; self.opened_at = 0
    def call(self, fn):
        if self.state == "OPEN":
            if time.time() - self.opened_at >= self.reset_timeout:
                self.state = "HALF_OPEN" # 일정 시간 경과 후 시험 가동
            else:
                raise RuntimeError("🔌 Circuit OPEN — 장애 지속 중으로 즉시 차단")
        try:
            r = fn(); self.fails = 0; self.state = "CLOSED"; return r
        except Exception:
            self.fails += 1
            if self.fails >= self.fail_threshold:
                self.state = "OPEN"; self.opened_at = time.time()
            raise

def retry(fn, max_retries=2, base=0.05):
    """지수 백오프와 지터를 적용한 재시도 로직"""
    for attempt in range(max_retries + 1):
        try:
            return fn()
        except Exception as e:
            if attempt == max_retries: raise
            delay = base * (2 ** attempt) * (1 + random.random())
            print(f"    (Retry) {attempt+1}차 시도 실패... {delay:.2f}s 대기")
            time.sleep(delay)

class ReliableAgent:
    """4대 신뢰성 패턴을 하나의 레이어로 통합한 에이전트"""
    def __init__(self, budget):
        self.budget = budget
        self.cb = CircuitBreaker()
        self.cache = {} # 멱등성 보장을 위한 결과 저장소

    def call_tool(self, name, fn, tokens=100, idem_key=None):
        # 패턴 1: 멱등성(Idempotency) - 동일 작업 중복 실행 방지
        if idem_key and idem_key in self.cache:
            print(f"  ♻️ {name}: 캐시된 결과 반환 (Idempotency)")
            return self.cache[idem_key]

        # 패턴 2: 예산(Budget) - 자원 소진 전 차단
        self.budget.charge(tokens)

        # 패턴 3 & 4: 회로 차단기 + 재시도 통합 호출
        result = self.cb.call(lambda: retry(fn))

        if idem_key:
            self.cache[idem_key] = result
        return result

# --- 통합 신뢰성 테스트 ---
calls_count = {"n": 0}
def flaky_api_demo():
    """처음 2번은 무조건 실패하는 불안정한 API 모사"""
    calls_count["n"] += 1
    if calls_count["n"] <= 2: raise ConnectionError("서버 연결 일시 오류")
    return "SUCCESS"

# 에이전트 설정 (최대 500토큰, 5스텝 예산)
agent = ReliableAgent(Budget(max_tokens=500, max_steps=5))

print("[1차 호출] 불안정한 도구 호출 시작 (Retry로 자동 복구)")
print(f"결과: {agent.call_tool('API', flaky_api_demo, tokens=50, idem_key='req_101')}\n")

print("[2차 호출] 동일 키로 재호출 (Idempotency 캐시 사용)")
print(f"결과: {agent.call_tool('API', flaky_api_demo, tokens=50, idem_key='req_101')}\n")

print(f"최종 통계: 사용 토큰={agent.budget.tokens}, 실제 함수 호출 횟수={calls_count['n']}")

[1차 호출] 불안정한 도구 호출 시작 (Retry로 자동 복구)
    (Retry) 1차 시도 실패... 0.08s 대기
    (Retry) 2차 시도 실패... 0.14s 대기
결과: SUCCESS

[2차 호출] 동일 키로 재호출 (Idempotency 캐시 사용)
  ♻️ API: 캐시된 결과 반환 (Idempotency)
결과: SUCCESS

최종 통계: 사용 토큰=50, 실제 함수 호출 횟수=3


### 5. 최종 통합 테스트: 신뢰성 있는 AI 비서 시나리오
아래 코드는 `ReliableAgent`를 사용하여 실제 도구 호출을 시뮬레이션합니다.

In [51]:
# === Upstage Solar 셋업 ===
from openai import OpenAI
from google.colab import userdata

try:
    api_key = userdata.get('UPSTAGE_API_KEY')
    client = OpenAI(api_key=api_key, base_url="https://api.upstage.ai/v1")
    MODEL = "solar-pro3"

    def ask(system, user, **kw):
        """Solar 단일 호출 헬퍼."""
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role":"system","content":system},{"role":"user","content":user}],
            **kw
        )
        return r.choices[0].message.content.strip()
    print("Solar 준비 완료:", MODEL)
except Exception as e:
    print("⚠️ UPSTAGE_API_KEY를 Secret에 설정해주세요:", e)

Solar 준비 완료: solar-pro3


In [52]:
# [모의 진행 코드]
# 1. 도구 정의 (실패 확률을 낮춘 버전)
def search_tool():
    # 10% 확률로만 실패하도록 조정하여 안정적인 테스트 보장
    if random.random() < 0.1:
        raise ConnectionError("검색 서버 응답 지연")
    return "서울의 오늘 날씨는 맑음입니다."

def send_email_tool(content):
    print(f"    [Email Service] '{content}' 발송 완료!")
    return "Email Sent OK"

# 2. 에이전트 초기화
my_budget = Budget(max_tokens=1000, max_steps=10)
reliable_ai = ReliableAgent(my_budget)

# 3. 시나리오 실행
try:
    print("--- Step 1: 정보 검색 (Retry 테스트) ---")
    info = reliable_ai.call_tool("Search", search_tool, tokens=150)
    print(f"검색 결과: {info}\n")

    print("--- Step 2: 이메일 발송 (Idempotency 테스트) ---")
    email_key = "task_email_final"
    res1 = reliable_ai.call_tool("Email", lambda: send_email_tool(info), tokens=100, idem_key=email_key)
    print(f"1차 발송 시도: {res1}")

    # 동일 작업 재시도 (캐시 작동 확인)
    res2 = reliable_ai.call_tool("Email", lambda: send_email_tool(info), tokens=100, idem_key=email_key)
    print(f"2차 발송 시도 (중복): {res2}\n")

    print("--- Step 3: 최종 예산 확인 ---")
    print(f"사용된 토큰: {reliable_ai.budget.tokens} / 남은 스텝: {reliable_ai.budget.max_steps - reliable_ai.budget.steps}")

except Exception as e:
    print(f"❌ 에이전트 실행 중단: {e}")

--- Step 1: 정보 검색 (Retry 테스트) ---
    (Retry) 1차 시도 실패... 0.08s 대기
    (Retry) 2차 시도 실패... 0.14s 대기
❌ 에이전트 실행 중단: 검색 서버 응답 지연


In [25]:
# === [실습] 실제 AI 호출이 포함된 통합 시나리오 ===

# 1. 에이전트 초기화 (실제 AI 호출용 예산 설정)
final_budget = Budget(max_tokens=2000, max_steps=10)
reliable_ai = ReliableAgent(final_budget)

# 2. 도구 정의
def ai_search_tool(query):
    # 실제 AI 호출을 도구화
    return ask("You are a helpful assistant.", f"Search result for: {query}")

def send_email_tool(content):
    print(f"    [Email Service] '{content[:20]}...' 발송 완료!")
    return "Email Sent OK"

# 3. 시나리오 실행
try:
    print("--- Step 1: 실제 AI 호출 (Reliability Layer 보호) ---")
    # 네트워크 지연 등을 고려해 retry가 감싸고 있음
    info = reliable_ai.call_tool(
        "Solar_Search",
        lambda: ai_search_tool("신뢰성 있는 에이전트 설계 패턴 4가지는?"),
        tokens=500
    )
    print(f"AI 답변: {info}\n")

    print("--- Step 2: 답변 내용 이메일 발송 (Idempotency 테스트) ---")
    email_key = "task_report_final_001"
    res1 = reliable_ai.call_tool("Email", lambda: send_email_tool(info), tokens=100, idem_key=email_key)
    print(f"1차 시도: {res1}")

    # 중복 실행 방지 확인
    res2 = reliable_ai.call_tool("Email", lambda: send_email_tool(info), tokens=100, idem_key=email_key)
    print(f"2차 시도 (중복): {res2}\n")

    print("--- Step 3: 최종 자원 상태 확인 ---")
    print(f"사용된 총 토큰: {reliable_ai.budget.tokens} / 남은 스텝: {reliable_ai.budget.max_steps - reliable_ai.budget.steps}")

except Exception as e:
    print(f"❌ 에이전트 실행 중단: {e}")

--- Step 1: 실제 AI 호출 (Reliability Layer 보호) ---
AI 답변: 신뢰성 있는 에이전트(Agent) 설계 시 고려되는 주요 패턴 4가지는 다음과 같습니다. 각 패턴은 시스템의 **내결함성(Fault Tolerance)**, **확장성(Scalability)**, **안정성(Reliability)**을 보장하는 데 초점을 맞춥니다.  

### 1. **서킷 브레이커 패턴 (Circuit Breaker Pattern)**  
   - **목적**: 장애 전파 방지 및 시스템 안정성 유지.  
   - **동작**:  
     - 에이전트 간 통신에서 실패 임계값(예: 연속 5회 실패) 도달 시 "서킷 개방" 상태로 전환.  
     - 일정 시간("쿨다운" 기간) 후 테스트 호출을 통해 서비스 복구 확인.  
   - **적용 예**: 마이크로서비스 간 API 호출, 외부 시스템 연동.  

### 2. **리트라이 패턴 (Retry Pattern with Exponential Backoff)**  
   - **목적**: 일시적 오류(네트워크 지연, 타임아웃) 자동 복구.  
   - **핵심**:  
     - 재시도 시 **지수 백오프(Exponential Backoff)** 적용 (예: 1초 → 2초 → 4초).  
     - 최대 재시도 횟수 및 재시도 간 임의 지연(Random Jitter) 추가.  
   - **주의점**: 멱등성(Idempotency) 보장 필요.  

### 3. **디드-온-디맨드 (Dead Man’s Switch / Health Check with Fallback)**  
   - **목적**: 에이전트 또는 외부 시스템의 장애 신속 감지 및 대체 처리.  
   - **구성**:  
     - **헬스 체크**: 주기적으로 "살아 있음" 신호(Heartbeat) 확인.  
     - **폴백(Fallback)**: 장애 시 기본값/캐시된 데이터 또는 대체 서비스 사용.  
   -

## 정리
- **시간/토큰 예산**은 비용·지연 폭주를 사전에 끊습니다(예측 가능한 리소스 소비).
- **샌드박스**는 생성된 코드의 위험(파일·import·무한 루프)을 격리합니다(보안).
- **통합 Reliability Layer**는 회로차단·재시도·멱등성·예산을 한 경로로 묶어 **불안정한 의존성 위에서도 안전한 에이전트**를 만듭니다.
- **Circuit Breaker**(차단) · **Retry+Jitter**(일시 실패) · **Budget**(폭주 방지) · **Idempotency**(중복 방지)가 4대 신뢰성 패턴.
- 자율 에이전트 + Reliability Layer = **Production-grade 에이전트**.

**NEXT:** 그럼에도 실패는 발생합니다 — 실패 Trace를 **분류하고 회귀셋으로 환류**해 에이전트를 진화시키는 방법을 배워봅니다.